In [ ]:
# imports
import torch
from model_trainer import load_model
from stock_dataloader import create_stock_dataloader
from metrics import direct_full_series, direct_category_window

In [ ]:
# load models
lstm_model = load_model('models\StockLSTM_ModelFull')
transformer_model = load_model('models\StockTransformer_ModelFull')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# load data

# Hyperparameters
SEQ_LEN = 1250          # default 1250; window for set of time-series data points
BATCH_SIZE = 128        # default 128; uses 25-30% of 12GB NVIDIA GeForce RTX 4070 GPU
STOCKS_PER_BUCKET = 13  # default 13; number of stocks per category bucket
TRAIN_PER_BUCKET = 10   # default 10; number of training stocks per category bucket

stock_csv = 'selected_stocks_data.csv'
metadata_csv = 'selected_stocks_quality.csv'
stock_dataloader = create_stock_dataloader(stock_csv, metadata_csv, seq_len=SEQ_LEN, batch_size=BATCH_SIZE,
                                           stocks_per_bucket=STOCKS_PER_BUCKET, train_per_bucket=TRAIN_PER_BUCKET)
eval_tickers = stock_dataloader['eval_tickers']
eval_stock_series = stock_dataloader['eval_raw_series']

In [ ]:
# evaluation metrics

seq_len = SEQ_LEN      # same as categorization length
stride = int(SEQ_LEN * 0.9)        # 90% of categorization window for smooth errors

results = {}
for ticker, series in eval_stock_series.items():
    series = series.unsqueeze(0)
    lstm_full_mse, lstm_full_mae = direct_full_series(lstm_model, series, seq_len, stride, device)
    lstm_cat_mse, lstm_cat_mae = direct_category_window(lstm_model, series, seq_len, device)
    transformer_full_mse, transformer_full_mae = direct_full_series(transformer_model, series, seq_len, stride, device)
    transformer_cat_mse, transformer_cat_mae = direct_category_window(transformer_model, series, seq_len, device)
    results[ticker] = {
        'lstm_full_mse': lstm_full_mse,
        'lstm_full_mae': lstm_full_mae,
        'lstm_cat_mse': lstm_cat_mse,
        'lstm_cat_mae': lstm_cat_mae,
        'trans_full_mse': transformer_full_mse,
        'trans_full_mae': transformer_full_mae,
        'trans_cat_mse': transformer_cat_mse,
        'trans_cat_mae': transformer_cat_mae,
    }

In [ ]:
for ticker, metrics in results.items():
    print(f"Ticker: {ticker}")
    for metric_name, value in metrics.items():
        print(f"  {metric_name}: {value:.6f}")
    print()